In [25]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import re
import json

In [26]:
SEEN_LOT_URLS = set()

In [27]:
def clean_auction_urls(df):
    """Убирает /overview из ссылок на аукционы"""
    df = df.copy()
    df['auction_url_clean'] = df['auction_url'].str.replace('/overview', '', regex=False)
    # Убираем дубликаты по очищенным ссылкам
    df = df.drop_duplicates(subset=['auction_url_clean'])
    print(f"Обработано {len(df)} уникальных аукционов")
    return df

In [28]:
def fetch_page(url, headers=None):
    """Загружает HTML страницы с заголовками браузера"""
    if headers is None:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.9',
            'Accept-Encoding': 'gzip, deflate',
            'Connection': 'keep-alive',
        }
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        return response.text
    except Exception as e:
        print(f"Ошибка загрузки {url}: {e}")
        return None

In [29]:
def parse_auction_lots_static(auction_url, auction_title):
    """Парсит лоты со страницы аукциона (статический парсинг)"""
    lots_data = []
    
    html = fetch_page(auction_url)
    if not html:
        return lots_data
    
    soup = BeautifulSoup(html, 'lxml')
    
    # === ИЗВЛЕКАЕМ ДАТУ АУКЦИОНА ===
    auction_date = 'Unknown'
    try:
        date_elem = soup.find(class_='seldon-sale-header-banner__date')
        if date_elem:
            auction_date = date_elem.get_text(strip=True)
            print(f"  📅 Дата аукциона: {auction_date}")
    except Exception as e:
        print(f"  ⚠ Не найдена дата аукциона: {e}")
    
    # === Ищем карточки лотов ===
    lot_cards = soup.find_all('div', class_='seldon-grid-item')
    print(f"  🎯 Найдено элементов seldon-grid-item: {len(lot_cards)}")
    
    # Фильтруем только те, что содержат ссылки на лоты 
    filtered_cards = []
    for card in lot_cards:
        link = card.find('a', href=re.compile(r'/detail/'))
        if link:
            filtered_cards.append(card)
    
    print(f" Найдено лотов с ссылками: {len(filtered_cards)}")
    
    for i, card in enumerate(filtered_cards):
        try:
            lot_info = {
                'auction_title': auction_title,
                'auction_url': auction_url,
                'auction_date': auction_date,
                'lot_url': None,
                'lot_number': None,
                'title': None,
                'artist': None,
                'estimate': None,
                'price': None,
                'currency': None,
                'image_url': None
            }
            
            # === Извлекаем ссылку на лот ===
            link = card.find('a', href=re.compile(r'/detail/'))
            if not link:
                continue
            
            lot_url = link.get('href')
            if lot_url.startswith('/'):
                lot_url = 'https://www.phillips.com' + lot_url
            
            lot_info['lot_url'] = lot_url
            
            # === Извлекаем номер лота из URL ===
            match = re.search(r'/detail/[^/]+/(\d+)', lot_info['lot_url'])
            if match:
                lot_info['lot_number'] = match.group(1)
            
            # === ИЗВЛЕКАЕМ ХУДОЖНИКА ===
            try:
                maker_elem = card.find(class_='seldon-text seldon-object-tile__maker seldon-text--headingSmall')
                if maker_elem:
                    artist_text = maker_elem.get_text(separator=' ', strip=True)
                    if artist_text:
                        lot_info['artist'] = artist_text
            except:
                pass
            
            # === ИЗВЛЕКАЕМ НАЗВАНИЕ РАБОТЫ ===
            try:
                title_elem = card.find(class_='seldon-text seldon-object-tile__title seldon-text--headingExtraSmall')
                if title_elem:
                    title_text = title_elem.get_text(separator=' ', strip=True)
                    if title_text:
                        lot_info['title'] = title_text
            except:
                pass
            
            # === Извлекаем эстимейт ===
            try:
                estimate_elem = card.find(string=re.compile(r'[\$£€HK\$]\s*[\d,]+'))
                if estimate_elem:
                    lot_info['estimate'] = estimate_elem.strip()
                else:
                    est_elem = card.find(class_=re.compile(r'estimate', re.I))
                    if est_elem:
                        lot_info['estimate'] = est_elem.get_text(strip=True)
            except:
                pass
            
            # === ИЗВЛЕКАЕМ ЦЕНУ ===
            try:
                # Ищем элемент с ценой продажи 
                price_div = card.find(class_='seldon-detail seldon-bid-snapshot__sold')
                if price_div:
                    price_span = price_div.find('span', class_='seldon-text seldon-text--labelSmall')
                    if price_span:
                        price_text = price_span.get_text(strip=True)
                        price_text = price_text.replace('\xa0', ' ').replace('&nbsp;', ' ')
                        
                        # Извлекаем валюту и сумму
                        match = re.search(r'([£€$HK\$])\s*([\d,]+\.?\d*)', price_text)
                        if match:
                            lot_info['currency'] = match.group(1)
                            lot_info['price'] = float(match.group(2).replace(',', ''))
            except:
                pass
            
            # === Извлекаем изображение ===
            try:
                img = card.find('img')
                if img:
                    img_src = img.get('src') or img.get('data-src')
                    if img_src and img_src.startswith('http'):
                        lot_info['image_url'] = img_src
            except:
                pass
            
            # Добавляем только если есть URL
            if lot_info['lot_url']:
                lots_data.append(lot_info)
                
        except Exception as e:
            print(f"    ⚠ Ошибка лота {i+1}: {e}")
            continue
    
    print(f"  Спаршено {len(lots_data)} лотов")
    return lots_data

In [30]:
def save_lots_to_csv(lots_data, filename='phillips_lots.csv'):
    """Сохраняет лоты в CSV"""
    if not lots_data:
        print("Нет данных для сохранения")
        return None
    
    df = pd.DataFrame(lots_data)
    
    columns_order = [
        'auction_title', 'auction_url', 'auction_date', 'lot_url', 'lot_number',
        'artist', 'title', 'estimate', 'price', 'currency', 'image_url'
    ]
    existing_cols = [c for c in columns_order if c in df.columns]
    df = df[existing_cols]
    df = df.drop_duplicates(subset=['lot_url'])
    
    df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\nСохранено {len(df)} лотов в {filename}")
    return df

In [7]:
if __name__ == "__main__":
    AUCTIONS_CSV = 'phillips_auctions.csv'
    OUTPUT_CSV = 'phillips_lots.csv'
    
    print("Запуск парсинга")
    print("="*60)
    
    # 1. Загружаем и чистим URLs
    df_auctions = pd.read_csv(AUCTIONS_CSV)
    df_auctions = clean_auction_urls(df_auctions)
    
    all_lots = []
    
    # 2. Парсим каждый аукцион (для теста — первые 3)
    for idx, row in df_auctions.iterrows():  # Уберать .head(3) для полного парсинга
        auction_title = row['auction_title']
        auction_url = row['auction_url_clean']
        
        print(f"\n🔹 [{idx+1}/{len(df_auctions)}] {auction_title}")
        print(f"  {auction_url}")
        
        lots = parse_auction_lots_static(auction_url, auction_title)
        all_lots.extend(lots)
        
        # Пауза между запросами
        time.sleep(2)
    
    # 3. Сохраняем результаты
    if all_lots:
        df_lots = save_lots_to_csv(all_lots, OUTPUT_CSV)
        
        # Предпросмотр
        if df_lots is not None and not df_lots.empty:
            print("\n" + "="*60)
            print("📋 ПРЕДПРОСМОТР")
            print("="*60)
            cols = ['auction_title', 'lot_number', 'artist', 'title', 'estimate']
            available = [c for c in cols if c in df_lots.columns]
            print(df_lots[available].head(10).to_string(index=False))
    else:
        print("Лоты не собраны")

Запуск парсинга
Обработано 353 уникальных аукционов

🔹 [1/353] MODERN & CONTEMPORARY ART
  https://www.phillips.com/auction/UK010326
  📅 Дата аукциона: 7 March 12pm GMT 2026
  🎯 Найдено элементов seldon-grid-item: 138
 Найдено лотов с ссылками: 133
  Спаршено 133 лотов

🔹 [2/353] MODERN & CONTEMPORARY ART EVENING SALE
  https://www.phillips.com/auction/UK010226
  📅 Дата аукциона: 5 March 4pm GMT 2026
  🎯 Найдено элементов seldon-grid-item: 32
 Найдено лотов с ссылками: 27
  Спаршено 27 лотов

🔹 [3/353] MODERN & CONTEMPORARY ART
  https://www.phillips.com/auction/NY010126
  📅 Дата аукциона: 28 February 11am ET 2026
  🎯 Найдено элементов seldon-grid-item: 167
 Найдено лотов с ссылками: 164
  Спаршено 164 лотов

🔹 [4/353] MODERN & CONTEMPORARY ART: ONLINE AUCTION, NEW YORK
  https://www.phillips.com/auction/NY010226
  📅 Дата аукциона: Mar 10 2026
  🎯 Найдено элементов seldon-grid-item: 187
 Найдено лотов с ссылками: 184
  Спаршено 184 лотов


KeyboardInterrupt: 

In [73]:
if __name__ == "__main__":
    #НАСТРОЙКИ ДИАПАЗОНА
    START_IDX = 340   #Начало диапазона 
    END_IDX = 350    #Конец диапазона 
    
    AUCTIONS_CSV = 'phillips_auctions.csv'
    OUTPUT_CSV = 'phillips_lots_350.csv'
    
    print(f"Запуск парсинга: аукционы [{START_IDX}:{END_IDX}]")
    print("="*60)
    
    #чистка URLs
    df_auctions = pd.read_csv(AUCTIONS_CSV)
    df_auctions = clean_auction_urls(df_auctions)
    
    #диапазон аукционов
    df_slice = df_auctions.iloc[START_IDX:END_IDX]
    print(f"Обрабатываем аукционы {START_IDX+1}–{END_IDX} из {len(df_auctions)}")
    
    all_lots = []
    success_count = 0
    blocked_count = 0
    
    for idx, row in df_slice.iterrows():
        auction_title = row['auction_title']
        auction_url = row['auction_url_clean']
        
        print(f"\n🔹 [{idx+1}] {auction_title[:50]}...")
        
        lots = parse_auction_lots_static(auction_url, auction_title)
        
        if lots:
            all_lots.extend(lots)
            success_count += 1
            print(f"   {len(lots)} лотов")
        else:
            blocked_count += 1
            print(f"   Пропущено (403 или нет лотов)")
        
        time.sleep(5)
    
    # Сохраняем результаты
    if all_lots:
        save_lots_to_csv(all_lots, OUTPUT_CSV)

Запуск парсинга: аукционы [340:350]
Обработано 353 уникальных аукционов
Обрабатываем аукционы 341–350 из 353

🔹 [341] CONTEMPORARY ART DAY SALE...
  📅 Дата аукциона: Oct 17 2013
  🎯 Найдено элементов seldon-grid-item: 191
 Найдено лотов с ссылками: 190
  Спаршено 190 лотов
   190 лотов

🔹 [342] CONTEMPORARY ART EVENING SALE...
  📅 Дата аукциона: Oct 16 2013
  🎯 Найдено элементов seldon-grid-item: 39
 Найдено лотов с ссылками: 38
  Спаршено 38 лотов
   38 лотов

🔹 [343] PADDLES ON...
  📅 Дата аукциона: Oct 10 2013
  🎯 Найдено элементов seldon-grid-item: 2
 Найдено лотов с ссылками: 0
  Спаршено 0 лотов
   Пропущено (403 или нет лотов)

🔹 [344] UNDER THE INFLUENCE...
  📅 Дата аукциона: Sep 19 2013
  🎯 Найдено элементов seldon-grid-item: 261
 Найдено лотов с ссылками: 260
  Спаршено 260 лотов
   260 лотов

🔹 [345] CONTEMPORARY ART DAY SALE...
  📅 Дата аукциона: Jun 28 2013
  🎯 Найдено элементов seldon-grid-item: 148
 Найдено лотов с ссылками: 147
  Спаршено 147 лотов
   147 лотов

🔹 [346]